# CBIS-DDSM — ResNet-50 Baseline (Breast Cancer Classification)

Baseline supervised classifier: malignant vs. not-malignant on full mammograms.

**Setup:** set `DATA_DIR` below to the folder that contains the `csv/` and `jpeg/` folders, then Run All.
Run the linking cell (Section 1) first and check that the printed image counts look right
(roughly ~1200+ train, ~360+ test) before launching training.

This notebook also saves the trained weights to `resnet50_baseline.pth` for reuse later
(Grad-CAM / attention-supervision stages).

In [ ]:
# ---------------------------- CONFIG ----------------------------
# Folder that directly contains the "csv" and "jpeg" subfolders:
DATA_DIR    = "/path/to/archive (1)"   # <-- EDIT THIS (e.g. your extracted Kaggle folder)

COHORT      = "Mass"   # "Mass" (default) ; calcification cases can be added later
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 15
LR          = 1e-4
VAL_FRAC    = 0.15
NUM_WORKERS = 2        # set to 0 if you hit multiprocessing errors on macOS
SEED        = 42
# ----------------------------------------------------------------

In [ ]:
import os, re, random
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

## 1. Link pathology labels to JPEG images
Uses `dicom_info.csv` (full-mammogram rows) to find the JPEG paths, and the case-description
CSV for the malignant/benign label, joined on patient + side + view.

In [ ]:
CSV_DIR = os.path.join(DATA_DIR, "csv")

def norm(df):
    df = df.copy(); df.columns = [c.strip().lower() for c in df.columns]; return df

def local_jpeg(image_path):
    ip = str(image_path).replace("\\", "/")
    rel = ip.split("jpeg/", 1)[1] if "jpeg/" in ip else os.path.basename(ip)
    return os.path.join(DATA_DIR, "jpeg", rel)

di = norm(pd.read_csv(os.path.join(CSV_DIR, "dicom_info.csv")))
full = di[di["seriesdescription"].astype(str).str.contains("full mammogram", case=False, na=False)].copy()
pat = full["patientid"].astype(str).str.extract(
    r"(?P<cohort>Mass|Calc)-(?P<split>Training|Test)_(?P<patient_id>P_\d+)_(?P<side>LEFT|RIGHT)_(?P<view>CC|MLO)")
full = pd.concat([full.reset_index(drop=True), pat.reset_index(drop=True)], axis=1)
full["jpeg"] = full["image_path"].apply(local_jpeg)

def build_split(case_csv, split):
    cc = norm(pd.read_csv(os.path.join(CSV_DIR, case_csv)))
    cc["label"] = (cc["pathology"].astype(str).str.upper() == "MALIGNANT").astype(int)
    lab = (cc.groupby(["patient_id", "left or right breast", "image view"], as_index=False)["label"].max()
             .rename(columns={"left or right breast": "side", "image view": "view"}))
    sub = full[(full["cohort"] == COHORT) & (full["split"] == split)]
    m = sub.merge(lab, on=["patient_id", "side", "view"], how="inner")
    m = m[m["jpeg"].apply(os.path.exists)]
    return m[["jpeg", "label", "patient_id", "side", "view"]].drop_duplicates("jpeg").reset_index(drop=True)

train_df = build_split("mass_case_description_train_set.csv", "Training")
test_df  = build_split("mass_case_description_test_set.csv",  "Test")

print("Full-mammogram rows in dicom_info:", len(full))
print(f"\nTRAIN images linked: {len(train_df)}")
print(train_df["label"].value_counts().rename({0:'not-malignant',1:'malignant'}))
print(f"\nTEST images linked: {len(test_df)}")
print(test_df["label"].value_counts().rename({0:'not-malignant',1:'malignant'}))

if len(train_df) == 0:
    print("\n[!] No images linked. Paste me this output to fix the linker:")
    print("dicom_info columns:", list(di.columns))
    print(full[["patientid","seriesdescription","image_path"]].head(3).to_dict("records"))

## 2. Datasets & DataLoaders

In [ ]:
mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
train_tfms = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tfms = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

class MammoDS(Dataset):
    def __init__(self, df, tfms): self.df = df.reset_index(drop=True); self.tfms = tfms
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return self.tfms(Image.open(row["jpeg"]).convert("L")), int(row["label"])

tr_df, val_df = train_test_split(train_df, test_size=VAL_FRAC,
                                 stratify=train_df["label"], random_state=SEED)
train_loader = DataLoader(MammoDS(tr_df, train_tfms), batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(MammoDS(val_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(MammoDS(test_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"train {len(tr_df)} | val {len(val_df)} | test {len(test_df)}")

## 3. Model — ImageNet-pretrained ResNet-50

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

## 4. Train (tracks validation AUC, keeps the best epoch)

In [ ]:
@torch.no_grad()
def eval_probs(loader):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(device)), 1)[:, 1].cpu().numpy()
        ps += p.tolist(); ys += y.tolist()
    return np.array(ys), np.array(ps)

best_auc, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        running += loss.item() * x.size(0)
    yv, pv = eval_probs(val_loader)
    auc = roc_auc_score(yv, pv)
    print(f"epoch {epoch:02d} | train loss {running/len(tr_df):.4f} | val AUC {auc:.4f}")
    if auc > best_auc:
        best_auc, best_state = auc, {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
torch.save(best_state, "resnet50_baseline.pth")
print(f"\nBest val AUC: {best_auc:.4f}  (weights saved to resnet50_baseline.pth)")

## 5. Evaluate on the held-out test set

In [ ]:
yt, pt = eval_probs(test_loader)
pred = (pt >= 0.5).astype(int)
print("=== Test metrics (for your supervisor) ===")
print(f"AUC:       {roc_auc_score(yt, pt):.3f}")
print(f"F1:        {f1_score(yt, pred):.3f}")
print(f"Precision: {precision_score(yt, pred):.3f}")
print(f"Recall:    {recall_score(yt, pred):.3f}")
print("\n", classification_report(yt, pred, target_names=["not-malignant","malignant"]))

fpr, tpr, _ = roc_curve(yt, pt)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(fpr, tpr, label=f"AUC={roc_auc_score(yt, pt):.3f}")
ax[0].plot([0,1],[0,1],"--",color="gray"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[0].set_title("ROC — ResNet-50 baseline"); ax[0].legend()
cm = confusion_matrix(yt, pred)
im = ax[1].imshow(cm, cmap="Blues"); ax[1].set_title("Confusion matrix")
ax[1].set_xticks([0,1]); ax[1].set_xticklabels(["not-malig","malig"])
ax[1].set_yticks([0,1]); ax[1].set_yticklabels(["not-malig","malig"])
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

## Notes
- This is the **baseline** — the contribution comes later (VAE anomaly detection and ROI-guided attention supervision).
- Labels are **binary** (malignant vs. not), folding `BENIGN_WITHOUT_CALLBACK` into not-malignant. Confirm this choice with Dr Florescu.
- The official CBIS-DDSM **train/test split** is respected; a small validation slice is carved from train only for monitoring.
- Class imbalance is present; for the baseline we keep plain cross-entropy. Class weighting / focal loss can be added next.
- `resnet50_baseline.pth` is saved for the Grad-CAM and attention stages.